# **Laboratorio 5: Detección de Movimiento y Seguimiento de Objetos** ⚙️🖼️

**Asignatura:** Visión por Ordenador I  
**Grado:** Ingeniería Matemática e Inteligencia Artificial  
**Curso:** 2026/2027  

### **Objetivo**
En esta parte de la práctica, aprenderá a implementar algoritmos de detección de movimiento mediante sustracción de fondo y flujo óptico, y explorará el filtro de Kalman para el seguimiento de objetos.

## **Librerías**

In [9]:
import cv2
import os
import numpy as np

## **Apartado B: Sustracción de Fondo**

### **Tarea B.1**: Carga de Vídeo
Cargue un vídeo en el cual se detectarán objetos en movimiento. Puede utilizar
un vídeo local o la cámara en tiempo real.

In [10]:
#TODO: Create a method that reads a video file (using VideoCapture from OpenCV) and returns its frames along with video properties
def read_video(videopath):
    """
    Reads a video file and returns its frames along with video properties.

    Args:
        videopath (str): The path to the video file.

    Returns:
        tuple: A tuple containing:
            - frames (list): A list of frames read from the video.
            - frame_width (int): The width of the video frames.
            - frame_height (int): The height of the video frames.
            - frame_rate (float): The frame rate of the video.
    """

    #TODO: Complete this line to read the video file
    cap = cv2.VideoCapture(videopath) 
    
    #TODO: Check if the video was successfully opened
    if not cap.isOpened():
        print('Error: Could not open the video file')

    #TODO: Get the szie of frames and the frame rate of the video
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) # Get the width of the video frames
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) # Get the height of the video frames
    frame_rate = cap.get(cv2.CAP_PROP_FPS) # Get the frame rate of the video
    
    #TODO: Use a loop to read the frames of the video and store them in a list
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames, frame_width, frame_height, frame_rate

#TODO: Path to the video file (visiontraffic.avi)
videopath = "../data/partB-D/visiontraffic.avi"

frames, frame_width, frame_height, frame_rate = read_video(videopath)

### **Tarea B.2**: Sustracción de Fondo mediante diferencia de frames
Realice una sustracción de fondo mediante diferencia de frames, para ello guarde
un frame con el fondo estático y úselo como frame de referencia de fondo.

In [11]:
#TODO:  Show the frames to select the reference frame, press 'n' to move to the next frame and 's' to select the frame
for i, frame in enumerate(frames):
    #TODO: Show the frame
    cv2.imshow('Video', frame)
    # Wait for the key
    key = cv2.waitKey(0)
    # If the key is 'n' continue to the next frame
    if key == ord('n'):
        continue
    # If the key is 's' select the frame as the reference frame
    elif key == ord('s'):
        #TODO: Copy the frame to use it as a reference
        reference_frame = frame.copy()
        #TODO: Convert the reference frame to grayscale
        reference_frame = cv2.cvtColor(reference_frame, cv2.COLOR_BGR2GRAY)
        print('Frame {} selected as reference frame'.format(i))
        break

cv2.destroyAllWindows()

#TODO: Compute the difference between the reference frame and the rest of the frames and show the difference
for frame in frames:
    # Convert the frame to grayscale
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    #TODO: Compute the difference between the reference frame and the current frame
    diff = cv2.absdiff(frame, reference_frame)
    cv2.imshow('Diferencia', diff)
    key = cv2.waitKey(1)
    if key == ord('q'):
        break

cv2.destroyAllWindows()

Frame 5 selected as reference frame


### **Tarea B.3**: Configuración de la sustracción de fondo con GMM
Configure el sustractor de fondo usando el modelo de mezcla de gaussianas
adaptativas (MOG2).

In [12]:
#TODO: Use MOG2 to detect the moving objects in the video

history = 100 # Number of frames to use to build the background model
varThreshold = 20  # Threshold to detect the background
detectShadows = True  # If True the algorithm detects the shadows

#TODO: Create the MOG2 object
mog2 = cv2.createBackgroundSubtractorMOG2(history,varThreshold,detectShadows)



In [13]:
print(frame_rate)

29.97002997002997


### **Tarea B.4**: Aplicación de la Sustracción de Fondo

Aplique la sustracción de fondo en cada frame para extraer los objetos en movimiento.

In [14]:
#TODO: Use a loop to detect the moving objects in the video using the MOG2 algorithm and 
# save a video storing the parameters at the name of the file

#TODO: Create a folder to store the videos
output_folder = "../results"
folder_path = os.path.join(output_folder, "video")
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

#TODO: Name of the output video file with the parameters (history, varThreshold, detectShadows)
videoname = f'/output_{history}_{varThreshold}_{detectShadows}.avi' # Name of the output video file with the parameters


#TODO: Create a VideoWriter object to save the video
fourcc = cv2.VideoWriter_fourcc(*"mp4v") # Codec to use
frame_size = (frame_width, frame_height) # Size of the frames
fps = frame_rate # Frame rate of the video
out = cv2.VideoWriter(folder_path+videoname, fourcc, fps, frame_size)

for frame in frames:
    #TODO: Apply the MOG2 algorithm to detect the moving objects
    mask = mog2.apply(frame)
    #TODO: Convert to BGR the mask to store it in the video
    mask = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    #TODO: Save the mask in a video
    out.write(mask)

out.release()

**Preguntas del Apartado B**
1. ¿Cómo afecta la variable `varThreshold` a la precisión de la detección?
2. ¿Qué ventajas presenta `createBackgroundSubtractorMOG2` frente a métodos simples de diferencia de imágenes?

## **Apartado C: Flujo Óptico**

### **Tarea C.1**: Configuración del Flujo Óptico

Consulte la documentación de `cv2.calcOpticalFlowPyrLK` para ver qué parametros se deben definir para realizar el cálculo del flujo óptico

In [15]:
#TODO: Use the method to read the video file (slow_traffic_small.mp4)
videopath = "../data/partB-D/slow_traffic_small.mp4"  # Path to the video file

frames, frame_width, frame_height, frame_rate = read_video(videopath)

#TODO: Define the parameters for Lucas-Kanade optical flow
winSize = (20, 20)
maxLevel = 2
criteria= (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)

### **Tarea C.2**: Detección de Puntos de Interés

Detecte los puntos de interés iniciales usando el algoritmo de Shi-Tomasi (`cv2.goodFeaturesToTrack`) en el primer frame.

In [16]:
prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)

mask = None
maxCorners = 100
qualityLevel = 0.3
minDistance = 7
blockSize = 7

p0 = cv2.goodFeaturesToTrack(prev_gray, mask=mask, maxCorners=maxCorners, qualityLevel=qualityLevel, minDistance=minDistance, blockSize=blockSize)

### **Tarea C.3**: Cálculo y Visualización del Flujo Óptico

In [17]:
#TODO: Use a loop to track the points of interest in the rest of the frames

# Create a mask image for drawing purposes
mask = np.zeros_like(frame)

for i, frame in enumerate(frames[1:]):
    input_frame = frame.copy()
    frame_gray = cv2.cvtColor(input_frame, cv2.COLOR_BGR2GRAY)

    p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, frame_gray, p0, None,winSize=winSize, maxLevel=maxLevel, criteria=criteria)

    good_new = p1[st == 1]
    good_old = p0[st == 1]

    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel().astype(int)
        c, d = old.ravel().astype(int)
        input_frame = cv2.circle(input_frame, (a, b), 5, (0, 0, 255), -1)
        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)

    prev_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)

    cv2.imshow('Frame', cv2.add(input_frame, mask))
    key = cv2.waitKey(1)
    if key == ord('q'):
        break

**Preguntas del Apartado C**
1. ¿Qué efecto tiene el parámetro `winSize` en la precisión del flujo óptico?
2. ¿Cómo influye el parámetro `qualityLevel` en la función `cv2.goodFeaturesToTrack` al detectar puntos de interés?

## **Apartado D: Filtro de Kalman para Seguimiento de Objetos**

### **Tarea D.1**: Configuración del Filtro de Kalman

Inicialice el filtro de Kalman (`cv2.KalmanFilter`) con una matriz de medición y transición adecuada para un seguimiento en dos dimensiones.

In [ ]:
#TODO: Use the method to read the video file (slow_traffic_small.mp4)
videopath = None  # Path to the video file

#TODO: Create the Kalman filter object
kf = cv2.KalmanFilter(None, None)
#TODO: Initialize the state of the Kalman filter
kf.measurementMatrix =  None # Measurement matrix np.array of shape (2, 4) and type np.float32
kf.transitionMatrix = None # Transition matrix np.array of shape (4, 4) and type np.float32
kf.processNoiseCov = None # Process noise covariance np.array of shape (4, 4) and type np.float32

measurement = np.array((2, 1), np.float32)
prediction = np.zeros((2, 1), np.float32)

#TODO: Show the frames to select the initial position of the object

for i, frame in enumerate(frames):
    # Show the frame
    cv2.imshow('Frame', frame)
    # Wait for the key
    key = cv2.waitKey(0)
    # If the key is 'n' continue to the next frame
    if key == ord('n'):
        continue
    # If the key is 's' select the position of the object
    elif key == ord('s'):
        # Select the position of the object
        x, y, w, h = cv2.selectROI('Frame', frame, False)
        track_window = (x, y, w, h)
        #TODO: Compute the center of the object
        cx = None
        cy = None
        #TODO: Initialize the state of the Kalman filter
        kf.statePost = np.array([[None], [None], [0], [0]], np.float32)

        # Initialize the covariance matrix
        kf.errorCovPost = np.eye(4, dtype=np.float32)
        
        #Predict the position of the object
        prediction = kf.predict()
        
        #TODO: Update the measurement and correct the Kalman filter
        measurement = np.array([[None], [None]], np.float32)
        kf.correct(measurement)

        #TODO: Crop the object
        crop = frame[None:None, None:None].copy()
        #TODO: Convert the cropped object to HSV
        hsv_crop = cv2.cvtColor()
        #TODO: Compute the histogram of the cropped object (Reminder: Use only the Hue channel (0-180))
        crop_hist = cv2.calcHist([None], [0], mask=None, histSize=[None], ranges=[None, None])
        cv2.normalize(crop_hist, crop_hist, 0, 255, cv2.NORM_MINMAX)
        
        print(f'Initial position selected: {x}, {y}')
        break

cv2.destroyAllWindows()

SystemError: <class 'cv2.KalmanFilter'> returned a result with an error set

: 

### **Tarea D.2**: Predicción y Corrección del Estado

Realice la predicción del estado y corrija la posición estimada en cada iteración.

In [ ]:
#TODO: Use the Kalman filter to predict the position of the points of interest

term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 1)

for frame in frames[i:]:
    #TODO: Copy the frame 
    input_frame = None
    #TODO: Convert the frame to HSV
    img_hsv = cv2.cvtColor()
    
    # Compute the back projection of the histogram
    img_bproject = cv2.calcBackProject([img_hsv], [0], crop_hist, [0, 180], 1)
    
    # Apply the mean shift algorithm to the back projection
    ret, track_window = cv2.meanShift(img_bproject, track_window, term_crit)
    x_,y_,w_,h_ = track_window
    #TODO: Compute the center of the object
    c_x = None
    c_y = None
    
    # Predict the position of the object
    prediction = kf.predict()

    #TODO: Update the measurement and correct the Kalman filter
    measurement = np.array([[None], [None]], np.float32)
    kf.correct(measurement)

    
    # Draw the predicted position
    cv2.circle(input_frame, (int(prediction[0][0]), int(prediction[1][0])), 5, (0, 0, 255), -1)
    cv2.circle(input_frame, (int(c_x), int(c_y)), 5, (0, 255, 0), -1)
    
    # Show the frame with the predicted position
    cv2.imshow('Frame', input_frame)
    key = cv2.waitKey(0)
    if key == ord('q'):
        break

cv2.destroyAllWindows()


**Preguntas del Apartado D**
1. ¿Cómo afecta el valor de `transitionMatrix` a la predicción en el filtro de Kalman?
2. ¿Cuál es la diferencia entre `measurementMatrix` y `transitionMatrix` en el contexto del seguimiento de objetos?